# Embeddings de palabras con Word2Vec

<a href="https://colab.research.google.com/github/noe-tec/machine-learning/blob/main/activities/activity_text_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Un *embedding* es un vector que representa algo, en este caso una palabra, de modo que las cosas parecidas quedan cerca. En esta actividad se usan los vectores de Word2Vec para medir similitudes, encontrar las palabras más parecidas y resolver analogías con aritmética de vectores.

En cada celda marcada con `TODO` hay que reemplazar `None` por el código correspondiente; la celda siguiente comprueba el resultado. No se requiere GPU.

In [ ]:
import gzip, time, urllib.request

import numpy as np

## Word2Vec

Word2Vec se entrena con una tarea sencilla: dada una palabra, distinguir las palabras que realmente aparecen a su alrededor de palabras tomadas al azar. Como consecuencia, dos palabras que suelen aparecer rodeadas de las mismas vecinas terminan con vectores parecidos.

Se usarán los vectores publicados por Google, entrenados con noticias en inglés (300 dimensiones). La siguiente celda descarga solo las 200,000 palabras más frecuentes, lo que toma unos segundos.

In [ ]:
URL_W2V = ("https://github.com/RaRe-Technologies/gensim-data/releases/download/"
           "word2vec-google-news-300/word2vec-google-news-300.gz")

def cargar_word2vec(url, limite):
    """Lee las primeras `limite` palabras del formato binario de Word2Vec."""
    with urllib.request.urlopen(url) as r, gzip.GzipFile(fileobj=r) as f:
        _, dim = map(int, f.readline().split())
        palabras, vectores = [], np.empty((limite, dim), dtype=np.float32)
        for i in range(limite):
            p = bytearray()
            while (c := f.read(1)) != b" ":
                if c != b"\n":
                    p += c
            palabras.append(p.decode("utf-8", errors="replace"))
            vectores[i] = np.frombuffer(f.read(4 * dim), dtype=np.float32)
    return palabras, vectores

inicio = time.time()
palabras, vectores = cargar_word2vec(URL_W2V, 200_000)
indice = {p: i for i, p in enumerate(palabras)}
print(f"{len(palabras):,} palabras de {vectores.shape[1]} dimensiones en {time.time() - inicio:.0f} s")

def vec(palabra):
    """Vector de una palabra (distingue mayúsculas: 'Paris', 'dog')."""
    return vectores[indice[palabra]]

### Similitud coseno

Para comparar dos vectores se usa el coseno del ángulo entre ellos:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}$$

Vale 1 si los vectores apuntan en la misma dirección y 0 si son perpendiculares. No depende de la longitud de los vectores, solo de su dirección.

In [ ]:
# TODO 1: implementa la similitud coseno entre dos vectores a y b.
# Sugerencia: np.dot y np.linalg.norm.
def similitud_coseno(a, b):
    return None

In [ ]:
# Comprobación del TODO 1
def comprobar_1():
    r = similitud_coseno(np.array([1.0, 0.0]), np.array([2.0, 0.0]))
    if r is None:
        print("Pendiente: similitud_coseno todavía regresa None.")
        return
    casos = [((1, 0), (2, 0), 1.0), ((1, 0), (0, 3), 0.0), ((1, 1), (-1, -1), -1.0)]
    for a, b, esperado in casos:
        obtenido = similitud_coseno(np.array(a, float), np.array(b, float))
        if not np.isclose(obtenido, esperado, atol=1e-6):
            print(f"Revisa: cos({a}, {b}) debería ser {esperado} y salió {obtenido:.3f}.")
            return
    print(f"Correcto. dog·cat = {similitud_coseno(vec('dog'), vec('cat')):.3f}, "
          f"dog·tractor = {similitud_coseno(vec('dog'), vec('tractor')):.3f}")
comprobar_1()

### Las palabras más parecidas

Buscar las palabras más parecidas a un vector requiere compararlo con las 200,000 palabras. Si todos los vectores se normalizan a longitud 1, la similitud coseno se reduce a un producto punto, y un solo producto matriz-vector calcula todas las similitudes a la vez.

In [ ]:
normalizados = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)   # [200000, 300]

# TODO 2: regresa las k palabras más parecidas al vector v, como lista de (palabra, similitud),
# ordenadas de mayor a menor similitud y omitiendo las palabras en `excluir`.
# Sugerencia: normaliza v, calcula normalizados @ v y ordena con np.argsort.
def mas_parecidas(v, k=5, excluir=()):
    return None

In [ ]:
# Comprobación del TODO 2
def comprobar_2():
    r = mas_parecidas(vec("dog"), k=5, excluir={"dog"})
    if r is None:
        print("Pendiente: mas_parecidas todavía regresa None.")
        return
    if len(r) != 5 or "dog" in [p for p, _ in r]:
        print("Revisa: deben ser 5 palabras y no debe aparecer 'dog'.")
        return
    if any(r[i][1] < r[i + 1][1] for i in range(4)):
        print("Revisa: las similitudes deben ir de mayor a menor.")
        return
    print("Correcto. Las más parecidas a 'dog':", [p for p, _ in r])
comprobar_2()

### Analogías

Si las relaciones entre palabras se codifican como direcciones en el espacio, la analogía «*a* es a *b* como *c* es a ?» se resuelve con aritmética de vectores: se toma el vector $\mathbf{b} - \mathbf{a} + \mathbf{c}$ y se busca la palabra más cercana.

In [ ]:
# TODO 3: resuelve la analogía «a es a b como c es a ?».
# Regresa las k palabras más parecidas a vec(b) - vec(a) + vec(c), excluyendo a, b y c.
def analogia(a, b, c, k=3):
    return None

In [ ]:
# Comprobación del TODO 3
def comprobar_3():
    r = analogia("man", "king", "woman")
    if r is None:
        print("Pendiente: analogia todavía regresa None.")
        return
    esperados = [(("man", "king", "woman"), "queen"), (("Spain", "Madrid", "France"), "Paris")]
    for args, esperado in esperados:
        obtenido = analogia(*args)[0][0]
        if obtenido != esperado:
            print(f"Revisa: {args} debería dar '{esperado}' y dio '{obtenido}'.")
            return
    print("Correcto. man : king :: woman :", analogia("man", "king", "woman"))
comprobar_3()

### Exploración en equipo

Usen la celda siguiente para buscar tres analogías: la que mejor funcione, la más extraña y la que muestre un sesgo. Algunas para empezar:

- `analogia("Japan", "Tokyo", "Mexico")`: ¿qué capital propone el modelo?
- `analogia("man", "computer_programmer", "woman")`: los vectores aprendieron de noticias escritas por personas, con sus sesgos.
- Las frases de varias palabras usan guion bajo: `New_York`, `ice_cream`.

Antes de compartir sus resultados, comparen con lo que pasa si **no** se excluyen las palabras de entrada: `mas_parecidas(vec("king") - vec("man") + vec("woman"))`.

In [ ]:
print(analogia("Japan", "Tokyo", "Mexico"))
print(analogia("man", "computer_programmer", "woman"))
print(mas_parecidas(vec("king") - vec("man") + vec("woman"), k=3))

## Preguntas

Respondan en esta celda, en una o dos oraciones cada una.

1. ¿Qué analogía sesgada encontraron? ¿De dónde proviene ese sesgo si nadie lo programó?
2. ¿Por qué es necesario excluir las palabras de entrada al resolver una analogía?

**Respuestas:**

1.
2.